# Virginia Piedmont native phenology

This notebook prototypes a pyinaturalist-first workflow for seasonal prevalence in the Virginia Piedmont.

Current assumptions:
- Region scope comes from 27 Piedmont counties and 9 independent cities passed as iNaturalist place IDs (matching the Virginia Physiographic Regions: Piedmont project's geographic scope).
- Nativity is evaluated against Virginia. First run after a scope change will make one API call per unique taxon via `annotate_taxon_nativity`; the 24-hour request cache absorbs repeat runs.
- Moths and butterflies are treated together as Lepidoptera.
- Life-stage slices are explicit for Lepidoptera: overall, larva, pupa, and adult.
- Bees (Anthophila) are included as a second focus group with an overall-only slice.

In [1]:
from pathlib import Path
import sys

import pandas as pd
import plotly.express as px

repo_root = Path.cwd()
if not (repo_root / 'helpers.py').exists() and (repo_root.parent.parent / 'helpers.py').exists():
    repo_root = repo_root.parent.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from helpers import annotate_taxon_nativity, get_inat_session, get_observation_rows

# 27 Piedmont counties + 9 independent cities (iNaturalist place IDs)
PIEDMONT_PLACE_IDS = [
    # Counties
    2913,   # Albemarle
    1660,   # Amelia
    1372,   # Appomattox
    1719,   # Arlington
    733,    # Brunswick
    2589,   # Buckingham
    617,    # Campbell
    1662,   # Charlotte
    2917,   # Culpeper
    2590,   # Cumberland
    738,    # Fairfax
    1494,   # Fauquier
    1721,   # Fluvanna
    2920,   # Goochland
    1075,   # Halifax
    1186,   # Henry
    3032,   # Louisa
    739,    # Loudoun
    1724,   # Lunenburg
    1493,   # Mecklenburg
    742,    # Nottoway
    743,    # Orange
    1664,   # Pittsylvania
    1491,   # Powhatan
    2923,   # Prince Edward
    744,    # Prince William
    2925,   # Spotsylvania
    # Independent cities
    13446,  # Alexandria
    13435,  # Charlottesville
    13434,  # Danville
    108238, # Fairfax (city)
    13451,  # Falls Church
    13483,  # Lynchburg
    13410,  # Manassas
    13430,  # Manassas Park
    13445,  # Martinsville
]
VA_PLACE_ID = 1297
LEPIDOPTERA_TAXON_ID = 47157
ANTHOPHILA_TAXON_ID = 630955
LIFESTAGE_TERM_ID = 1
LIFESTAGE_ADULT = 2
LIFESTAGE_PUPA = 4
LIFESTAGE_LARVA = 6
START = '2000-01-01'
END = None
PER_PAGE = 200
MAX_PAGES = 30

session = get_inat_session()

In [4]:
# -- Tier 1 preflight: cheap all-time overview + truncation gate --
import plotly.graph_objects as go

_place_id_str = ','.join(str(x) for x in PIEDMONT_PLACE_IDS)
_max_fetchable = PER_PAGE * MAX_PAGES
_any_truncated = False

# 1. Count-only checks (1 request per focus group, no pagination)
for _label, _taxon_id in [('Lepidoptera', LEPIDOPTERA_TAXON_ID), ('Anthophila', ANTHOPHILA_TAXON_ID)]:
    _r = session.get(
        'https://api.inaturalist.org/v1/observations',
        params={'place_id': _place_id_str, 'taxon_id': _taxon_id, 'per_page': 0},
    )
    _total = _r.json().get('total_results', None)
    print(f'{_label}: {_total:,} total observations across Piedmont places')
    if _total is not None and _total > _max_fetchable:
        _pct = _max_fetchable / _total * 100
        print(f'  ⚠ Tier 2 will fetch only {_pct:.1f}% ({_max_fetchable:,} of {_total:,})')
        _any_truncated = True
    else:
        print(f'  ✓ Volume OK (max fetchable {_max_fetchable:,})')

# 2. All-time weekly histograms (1 request each, no date filter)
_weeks = list(range(1, 53))
_month_ticks = [1, 5, 9, 14, 18, 22, 27, 31, 35, 40, 44, 48]
_month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

_lep_r = session.get(
    'https://api.inaturalist.org/v1/observations/histogram',
    params={'place_id': _place_id_str, 'taxon_id': LEPIDOPTERA_TAXON_ID, 'interval': 'week_of_year'},
)
_bee_r = session.get(
    'https://api.inaturalist.org/v1/observations/histogram',
    params={'place_id': _place_id_str, 'taxon_id': ANTHOPHILA_TAXON_ID, 'interval': 'week_of_year'},
)
_lep_hist = _lep_r.json().get('results', {}).get('week_of_year', {})
_bee_hist = _bee_r.json().get('results', {}).get('week_of_year', {})

_fig = go.Figure()
_fig.add_scatter(
    x=_weeks,
    y=[_lep_hist.get(str(w), 0) for w in _weeks],
    mode='lines',
    name='Lepidoptera',
)
_fig.add_scatter(
    x=_weeks,
    y=[_bee_hist.get(str(w), 0) for w in _weeks],
    mode='lines',
    name='Anthophila',
)
_fig.update_layout(
    title='All-time weekly observation counts across Piedmont places',
    xaxis=dict(tickvals=_month_ticks, ticktext=_month_names),
    yaxis_title='Observations (all time)',
)
_fig.show()

# 3. Truncation gate
print()
if _any_truncated:
    print('⚠ TRUNCATION WARNING: At least one focus group exceeds the Tier 2 page cap.')
    print()
    print('  Options before running the next cell:')
    print(f'    • Raise MAX_PAGES (currently {MAX_PAGES})')
    print(f'    • Narrow START (currently {START})')
    print(f'    • Accept truncation (Tier 2 fetches the most recent {_max_fetchable:,} per query)')
else:
    print('✓ All focus groups fit within Tier 2 limits — safe to proceed.')

Lepidoptera: 211,876 total observations across Piedmont places
  ⚠ Tier 2 will fetch only 2.8% (6,000 of 211,876)
Anthophila: 36,748 total observations across Piedmont places
  ⚠ Tier 2 will fetch only 16.3% (6,000 of 36,748)



⚠ TRUNCATION WARNING: At least one focus group exceeds the Tier 2 page cap.

  Options before running the next cell:
    • Raise MAX_PAGES (currently 30)
    • Narrow START (currently 2000-01-01)
    • Accept truncation (Tier 2 fetches the most recent 6,000 per query)


In [ ]:
import plotly.graph_objects as go

query_specs = [
    {
        'focus_group': 'Lepidoptera',
        'life_stage_bucket': 'overall',
        'taxa_filters': {'taxon_id': LEPIDOPTERA_TAXON_ID},
    },
    {
        'focus_group': 'Lepidoptera',
        'life_stage_bucket': 'larva',
        'taxa_filters': {
            'taxon_id': LEPIDOPTERA_TAXON_ID,
            'term_id': LIFESTAGE_TERM_ID,
            'term_value_id': LIFESTAGE_LARVA,
        },
    },
    {
        'focus_group': 'Lepidoptera',
        'life_stage_bucket': 'pupa',
        'taxa_filters': {
            'taxon_id': LEPIDOPTERA_TAXON_ID,
            'term_id': LIFESTAGE_TERM_ID,
            'term_value_id': LIFESTAGE_PUPA,
        },
    },
    {
        'focus_group': 'Lepidoptera',
        'life_stage_bucket': 'adult',
        'taxa_filters': {
            'taxon_id': LEPIDOPTERA_TAXON_ID,
            'term_id': LIFESTAGE_TERM_ID,
            'term_value_id': LIFESTAGE_ADULT,
        },
    },
    {
        'focus_group': 'Anthophila',
        'life_stage_bucket': 'overall',
        'taxa_filters': {'taxon_id': ANTHOPHILA_TAXON_ID},
    },
]

frames = []
for spec in query_specs:
    frame = get_observation_rows(
        kind='any',
        places=PIEDMONT_PLACE_IDS,
        taxa_filters=spec['taxa_filters'],
        d1=START,
        d2=END,
        per_page=PER_PAGE,
        max_pages=MAX_PAGES,
        session=session,
    )
    frame['focus_group'] = spec['focus_group']
    frame['life_stage_bucket'] = spec['life_stage_bucket']
    frames.append(frame)

phenology = pd.concat(frames, ignore_index=True)
phenology = annotate_taxon_nativity(phenology, nativity_place_id=VA_PLACE_ID, session=session)
native_phenology = phenology[phenology['nativity'].isin(['Native', 'Endemic'])].copy()
native_phenology['observed_on'] = pd.to_datetime(native_phenology['observed_on'])
native_phenology['month'] = native_phenology['observed_on'].dt.month
native_phenology['month_name'] = native_phenology['observed_on'].dt.strftime('%b')
native_phenology['week_of_year'] = native_phenology['observed_on'].dt.isocalendar().week.astype(int)

# Week-of-year line plot with month x-axis labels
_weeks = list(range(1, 53))
_month_ticks = [1, 5, 9, 14, 18, 22, 27, 31, 35, 40, 44, 48]
_month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

_weekly = (
    native_phenology.groupby(['focus_group', 'life_stage_bucket', 'week_of_year'])['obs_id']
    .nunique()
    .rename('observation_count')
    .reset_index()
)

_fig = go.Figure()
for (fg, lsb), grp in _weekly.groupby(['focus_group', 'life_stage_bucket']):
    grp = grp.set_index('week_of_year').reindex(_weeks, fill_value=0)
    _fig.add_scatter(
        x=_weeks,
        y=grp['observation_count'].values,
        mode='lines',
        name=f'{fg} – {lsb}',
    )
_fig.update_layout(
    title='Weekly native observation phenology (VA Piedmont)',
    xaxis=dict(tickvals=_month_ticks, ticktext=_month_names),
    yaxis_title='Unique observations',
)
_fig.show()

native_phenology[
    ['obs_id', 'focus_group', 'taxon_name', 'taxon_preferred_common_name', 'life_stage_bucket', 'nativity', 'observed_on']
].head()

,obs_id,focus_group,taxon_name,taxon_preferred_common_name,life_stage_bucket,nativity,observed_on
6,145608833,Lepidoptera,Antheraea polyphemus,Polyphemus Moth,overall,Native,2023-01-01
9,145719388,Lepidoptera,Scolecocampa liburna,Deadwood Borer Moth,overall,Native,2023-01-02
11,145716494,Lepidoptera,Ennominae,NaN,overall,Native,2023-01-02
12,185734425,Lepidoptera,Chlosyne nycteis,Silvery Checkerspot,overall,Native,2023-01-02
13,146251336,Lepidoptera,Phigalia denticulata,Toothed Phigalia Moth,overall,Native,2023-01-03


In [5]:
monthly_taxa = (
    native_phenology.groupby(
        [
            'focus_group',
            'life_stage_bucket',
            'month',
            'month_name',
            'taxon_id',
            'taxon_name',
            'taxon_preferred_common_name',
        ],
        dropna=False,
    )['obs_id']
    .nunique()
    .rename('observation_count')
    .reset_index()
)

poster_candidates = (
    native_phenology.groupby(
        ['focus_group', 'life_stage_bucket', 'taxon_id', 'taxon_name', 'taxon_preferred_common_name', 'nativity'],
        dropna=False,
    )
    .agg(
        observation_count=('obs_id', 'nunique'),
        months_present=('month', 'nunique'),
        first_photo=('photo_url', 'first'),
    )
    .reset_index()
    .sort_values(
        ['focus_group', 'life_stage_bucket', 'observation_count', 'months_present'],
        ascending=[True, True, False, False],
    )
)

poster_candidates.head(30)

,focus_group,life_stage_bucket,taxon_id,taxon_name,taxon_preferred_common_name,nativity,observation_count,months_present,first_photo
17,Anthophila,overall,118970,Bombus impatiens,Common Eastern Bumble Bee,Native,865,10,https://inaturalist-open-data.s3.amazonaws.com...
2,Anthophila,overall,51110,Xylocopa virginica,Eastern Carpenter Bee,Native,630,10,https://inaturalist-open-data.s3.amazonaws.com...
5,Anthophila,overall,52779,Bombus bimaculatus,Two-spotted Bumble Bee,Native,271,8,https://inaturalist-open-data.s3.amazonaws.com...
18,Anthophila,overall,120215,Bombus griseocollis,Brown-belted Bumble Bee,Native,267,8,https://inaturalist-open-data.s3.amazonaws.com...
12,Anthophila,overall,82523,Augochlora pura,Pure Green Sweat bee,Native,229,11,https://inaturalist-open-data.s3.amazonaws.com...
10,Anthophila,overall,57678,Lasioglossum,NaN,Native,138,9,https://inaturalist-open-data.s3.amazonaws.com...
7,Anthophila,overall,53648,Nomada,Nomad Bees,Native,131,5,https://inaturalist-open-data.s3.amazonaws.com...
9,Anthophila,overall,57677,Halictus,Furrow Bees,Native,130,7,https://inaturalist-open-data.s3.amazonaws.com...
8,Anthophila,overall,56887,Bombus pensylvanicus,American Bumble Bee,Native,113,7,https://inaturalist-open-data.s3.amazonaws.com...
30,Anthophila,overall,155108,Ceratina,Small Carpenter Bees,Native,96,8,https://inaturalist-open-data.s3.amazonaws.com...


In [ ]:
top_taxa = poster_candidates.groupby(['focus_group', 'life_stage_bucket']).head(4)['taxon_id']
plot_frame = monthly_taxa[monthly_taxa['taxon_id'].isin(top_taxa)].copy()
month_order = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
stage_order = ['overall', 'adult', 'pupa', 'larva']
plot_frame['month_name'] = pd.Categorical(plot_frame['month_name'], categories=month_order, ordered=True)
plot_frame['life_stage_bucket'] = pd.Categorical(plot_frame['life_stage_bucket'], categories=stage_order, ordered=True)
plot_frame = plot_frame.sort_values(['focus_group', 'life_stage_bucket', 'taxon_name', 'month'])

fig = px.line(
    plot_frame,
    x='month_name',
    y='observation_count',
    color='taxon_preferred_common_name',
    facet_row='focus_group',
    facet_col='life_stage_bucket',
    markers=True,
    hover_data=['taxon_name'],
    title='Monthly prevalence of selected native Lepidoptera and Anthophila (VA Piedmont counties)',
)
fig.update_yaxes(matches=None)
fig.show()